In [1]:
%pip install -q pgembed psycopg2-binary sqlalchemy pandas kagglehub

import os
import pandas as pd
import pgembed as pgserver
from sqlalchemy import create_engine, text
import kagglehub

pg = pgserver.get_server('/tmp/hw4_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri(), future=True)

with engine.connect() as conn:
    print(conn.execute(text('SELECT version()')).scalar_one())

# Варіант А: якщо Kaggle credentials налаштовані
dataset_dir = kagglehub.dataset_download('olistbr/brazilian-ecommerce')
print("Dataset directory:", dataset_dir)

PostgreSQL 17.9 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 14.2.1 20250110 (Red Hat 14.2.1-11), 64-bit
Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
Dataset directory: /kaggle/input/brazilian-ecommerce


In [2]:
files = {
    'olist_customers_raw': 'olist_customers_dataset.csv',
    'olist_orders_raw': 'olist_orders_dataset.csv',
    'olist_order_items_raw': 'olist_order_items_dataset.csv',
    'olist_products_raw': 'olist_products_dataset.csv',
    'olist_sellers_raw': 'olist_sellers_dataset.csv',
    'olist_order_reviews_raw': 'olist_order_reviews_dataset.csv',
}

for table_name, file_name in files.items():
    path = os.path.join(dataset_dir, file_name)
    df = pd.read_csv(path)
    df.to_sql(table_name, engine, if_exists='replace', index=False, chunksize=10_000)
    print(table_name, df.shape)

olist_customers_raw (99441, 5)
olist_orders_raw (99441, 8)
olist_order_items_raw (112650, 7)
olist_products_raw (32951, 9)
olist_sellers_raw (3095, 4)
olist_order_reviews_raw (99224, 7)


In [3]:
create_schema_query = text("""
DROP TABLE IF EXISTS customer_segments    CASCADE;
DROP TABLE IF EXISTS seller_score         CASCADE;
DROP TABLE IF EXISTS seller_alerts        CASCADE;
DROP TABLE IF EXISTS olist_customer_dim   CASCADE;
DROP TABLE IF EXISTS olist_order_reviews  CASCADE;
DROP TABLE IF EXISTS olist_order_items    CASCADE;
DROP TABLE IF EXISTS olist_orders         CASCADE;
DROP TABLE IF EXISTS olist_customers      CASCADE;
DROP TABLE IF EXISTS olist_products       CASCADE;
DROP TABLE IF EXISTS olist_sellers        CASCADE;

CREATE TABLE olist_customers (
    customer_id              TEXT PRIMARY KEY,
    customer_unique_id       TEXT NOT NULL,
    customer_zip_code_prefix INTEGER,
    customer_city            TEXT,
    customer_state           CHAR(2) NOT NULL
);
CREATE INDEX idx_olist_customers_unique_id ON olist_customers (customer_unique_id);

CREATE TABLE olist_orders (
    order_id                      TEXT PRIMARY KEY,
    customer_id                   TEXT NOT NULL REFERENCES olist_customers(customer_id),
    order_status                  TEXT NOT NULL,
    order_purchase_timestamp      TIMESTAMP NOT NULL,
    order_approved_at             TIMESTAMP,
    order_delivered_carrier_date  TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP,
    CHECK (order_status IN (
        'created', 'approved', 'invoiced', 'processing',
        'shipped', 'delivered', 'unavailable', 'canceled'
    ))
);

CREATE TABLE olist_products (
    product_id                 TEXT PRIMARY KEY,
    product_category_name      TEXT,
    product_name_length        INTEGER CHECK (product_name_length IS NULL OR product_name_length >= 0),
    product_description_length INTEGER CHECK (product_description_length IS NULL OR product_description_length >= 0),
    product_photos_qty         INTEGER CHECK (product_photos_qty IS NULL OR product_photos_qty >= 0),
    product_weight_g           INTEGER CHECK (product_weight_g IS NULL OR product_weight_g >= 0),
    product_length_cm          INTEGER CHECK (product_length_cm IS NULL OR product_length_cm >= 0),
    product_height_cm          INTEGER CHECK (product_height_cm IS NULL OR product_height_cm >= 0),
    product_width_cm           INTEGER CHECK (product_width_cm IS NULL OR product_width_cm >= 0)
);

CREATE TABLE olist_sellers (
    seller_id              TEXT PRIMARY KEY,
    seller_zip_code_prefix INTEGER,
    seller_city            TEXT,
    seller_state           CHAR(2) NOT NULL
);

CREATE TABLE olist_order_items (
    order_id            TEXT NOT NULL REFERENCES olist_orders(order_id),
    order_item_id       INTEGER NOT NULL,
    product_id          TEXT REFERENCES olist_products(product_id),
    seller_id           TEXT REFERENCES olist_sellers(seller_id),
    shipping_limit_date TIMESTAMP,
    price               NUMERIC(12, 2) CHECK (price IS NULL OR price >= 0),
    freight_value       NUMERIC(12, 2) CHECK (freight_value IS NULL OR freight_value >= 0),
    PRIMARY KEY (order_id, order_item_id)
);
CREATE INDEX idx_olist_order_items_seller_order ON olist_order_items (seller_id, order_id);

CREATE TABLE olist_order_reviews (
    review_row_id            BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    review_id                TEXT,
    order_id                 TEXT NOT NULL REFERENCES olist_orders(order_id),
    review_score             SMALLINT CHECK (review_score BETWEEN 1 AND 5),
    review_comment_title     TEXT,
    review_comment_message   TEXT,
    review_creation_date     TIMESTAMP,
    review_answer_timestamp  TIMESTAMP
);
CREATE INDEX idx_olist_order_reviews_order_id ON olist_order_reviews (order_id);
""")

with engine.begin() as conn:
    conn.execute(create_schema_query)
print("Typed schema created successfully!")

Typed schema created successfully!


In [4]:
insert_data_query = text("""
INSERT INTO olist_customers
SELECT
    customer_id,
    customer_unique_id,
    customer_zip_code_prefix::INTEGER,
    customer_city,
    customer_state::CHAR(2)
FROM olist_customers_raw;

INSERT INTO olist_orders
SELECT
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp::TIMESTAMP,
    order_approved_at::TIMESTAMP,
    order_delivered_carrier_date::TIMESTAMP,
    order_delivered_customer_date::TIMESTAMP,
    order_estimated_delivery_date::TIMESTAMP
FROM olist_orders_raw;

INSERT INTO olist_products
SELECT
    product_id,
    product_category_name,
    product_name_lenght::INTEGER,
    product_description_lenght::INTEGER,
    product_photos_qty::INTEGER,
    product_weight_g::INTEGER,
    product_length_cm::INTEGER,
    product_height_cm::INTEGER,
    product_width_cm::INTEGER
FROM olist_products_raw;

INSERT INTO olist_sellers
SELECT
    seller_id,
    seller_zip_code_prefix::INTEGER,
    seller_city,
    seller_state::CHAR(2)
FROM olist_sellers_raw;

INSERT INTO olist_order_items
SELECT
    order_id,
    order_item_id::INTEGER,
    product_id,
    seller_id,
    shipping_limit_date::TIMESTAMP,
    price::NUMERIC(12, 2),
    freight_value::NUMERIC(12, 2)
FROM olist_order_items_raw;

INSERT INTO olist_order_reviews (
    review_id, order_id, review_score, review_comment_title,
    review_comment_message, review_creation_date, review_answer_timestamp
)
SELECT
    review_id,
    order_id,
    review_score::SMALLINT,
    review_comment_title,
    review_comment_message,
    review_creation_date::TIMESTAMP,
    review_answer_timestamp::TIMESTAMP
FROM olist_order_reviews_raw;
""")

with engine.begin() as conn:
    conn.execute(insert_data_query)

smoke_test_query = """
SELECT 'customers' AS table_name, COUNT(*) AS n_rows FROM olist_customers
UNION ALL SELECT 'orders', COUNT(*) FROM olist_orders
UNION ALL SELECT 'order_items', COUNT(*) FROM olist_order_items
UNION ALL SELECT 'products', COUNT(*) FROM olist_products
UNION ALL SELECT 'sellers', COUNT(*) FROM olist_sellers
UNION ALL SELECT 'reviews', COUNT(*) FROM olist_order_reviews
ORDER BY table_name;
"""
print(pd.read_sql(smoke_test_query, engine))

    table_name  n_rows
0    customers   99441
1  order_items  112650
2       orders   99441
3     products   32951
4      reviews   99224
5      sellers    3095


In [5]:
query_2_1 = text("""
DROP TABLE IF EXISTS seller_alerts CASCADE;
CREATE TABLE seller_alerts (
    alert_id   BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    seller_id  TEXT NOT NULL REFERENCES olist_sellers(seller_id),
    alert_type TEXT NOT NULL CHECK (alert_type IN ('late_delivery', 'low_rating', 'data_quality')),
    severity   SMALLINT NOT NULL CHECK (severity BETWEEN 1 AND 5),
    details    TEXT,
    created_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
);

WITH seller_orders AS (
    SELECT DISTINCT
        oi.seller_id,
        oi.order_id
    FROM olist_order_items AS oi
),
late_sellers AS (
    SELECT
        so.seller_id,
        COUNT(DISTINCT o.order_id) AS late_orders
    FROM seller_orders AS so
    JOIN olist_orders AS o
        ON o.order_id = so.order_id
    WHERE o.order_delivered_customer_date IS NOT NULL
      AND o.order_estimated_delivery_date IS NOT NULL
      AND o.order_delivered_customer_date::DATE > o.order_estimated_delivery_date::DATE
    GROUP BY so.seller_id
    ORDER BY late_orders DESC, so.seller_id
    LIMIT 1
)
INSERT INTO seller_alerts (seller_id, alert_type, severity, details)
SELECT
    seller_id,
    'late_delivery',
    4,
    'Seller has ' || late_orders || ' late delivered orders in the loaded dataset'
FROM late_sellers
RETURNING alert_id, seller_id, alert_type, severity, created_at;
""")

with engine.begin() as conn:
    result = pd.read_sql(query_2_1, conn)
print(result)

   alert_id                         seller_id     alert_type  severity  \
0         1  4a3ca9315b744ce9f8e9374361493884  late_delivery         4   

                        created_at  
0 2026-09-27 12:34:39.649518+00:00  


**Пояснення 2.1:**  
Використання `RETURNING` дозволяє одразу після вставки нового рядка повернути його ключові поля (включно з автозгенерованим `alert_id` та `created_at`). Це усуває необхідність писати окремий `SELECT`-запит для перевірки результату і слугує автоматичним аудит-логом прямо у виводі (output) зошита.

In [6]:
query_2_2 = text("""
DROP TABLE IF EXISTS seller_score CASCADE;
CREATE TABLE seller_score (
    seller_id   TEXT PRIMARY KEY REFERENCES olist_sellers(seller_id),
    avg_rating  NUMERIC(3, 2),
    n_reviews   INTEGER NOT NULL DEFAULT 0 CHECK (n_reviews >= 0),
    tier        TEXT CHECK (tier IN ('gold', 'silver', 'bronze')),
    updated_at  TIMESTAMPTZ NOT NULL DEFAULT NOW()
);

WITH seller_orders AS (
    SELECT DISTINCT
        oi.seller_id,
        oi.order_id
    FROM olist_order_items AS oi
),
review_per_order AS (
    SELECT
        rv.order_id,
        AVG(rv.review_score)::NUMERIC(3, 2) AS order_review_score
    FROM olist_order_reviews AS rv
    WHERE rv.review_score IS NOT NULL
    GROUP BY rv.order_id
),
seller_rating AS (
    SELECT
        so.seller_id,
        AVG(rpo.order_review_score)::NUMERIC(3, 2) AS avg_rating,
        COUNT(rpo.order_id) AS n_reviews
    FROM seller_orders AS so
    LEFT JOIN review_per_order AS rpo
        ON rpo.order_id = so.order_id
    GROUP BY so.seller_id
)
INSERT INTO seller_score (seller_id, avg_rating, n_reviews, tier)
SELECT
    seller_id,
    avg_rating,
    n_reviews,
    CASE
        WHEN avg_rating IS NULL THEN NULL
        WHEN avg_rating >= 4.5 THEN 'gold'
        WHEN avg_rating >= 3.5 THEN 'silver'
        ELSE 'bronze'
    END AS tier
FROM seller_rating
ON CONFLICT (seller_id) DO UPDATE
SET avg_rating = EXCLUDED.avg_rating,
    n_reviews  = EXCLUDED.n_reviews,
    tier       = EXCLUDED.tier,
    updated_at = NOW()
RETURNING seller_id, avg_rating, n_reviews, tier, updated_at;
""")

with engine.begin() as conn:
    result = pd.read_sql(query_2_2, conn)
print(result.head())

                          seller_id  avg_rating  n_reviews    tier  \
0  a61cc04793308395a840807104365121        5.00          1    gold   
1  6d2f2e3b539480db1e0842b3a4e32e6e        3.67          3  silver   
2  c53bcd3be457a342a97e39e5a9f0be22        3.20          5  bronze   
3  9591fc341b1bfb7ef561e2968ec6e011        1.00          1  bronze   
4  f9ec7093df3a7b346b7bcf7864069ca3        4.78          9    gold   

                        updated_at  
0 2026-09-27 12:34:39.982996+00:00  
1 2026-09-27 12:34:39.982996+00:00  
2 2026-09-27 12:34:39.982996+00:00  
3 2026-09-27 12:34:39.982996+00:00  
4 2026-09-27 12:34:39.982996+00:00  


**Пояснення 2.2:**
1. **Чому `SELECT DISTINCT` у `seller_orders`:** Таблиця `order_items` містить окремі рядки для кожної позиції в замовленні. Якщо продавець продав два товари в одному замовленні, він з'явиться двічі. `DISTINCT` згортає це до однієї пари "продавець-замовлення", щоб не завищувати штучно метрики активності.
2. **Чому review потрібно агрегувати на рівні `order_id`:** Відгуки в Olist прив'язані до всього замовлення, а не до конкретного товару. Якщо не зробити попередню агрегацію, один відгук продублюється для кожної позиції товару в замовленні, що спотворить середню оцінку продавця. (Також варто зазначити обмеження датасету: якщо в замовленні були товари різних продавців, загальний відгук за замовлення рівноцінно вплине на рейтинг усіх цих продавців).
3. **Ідемпотентність через `ON CONFLICT`:** Якщо запустити скрипт вдруге, конструкція `ON CONFLICT DO UPDATE` не намагатиметься створити дублікат (що викликало б помилку PRIMARY KEY), а безпечно оновить існуючі записи новими розрахованими значеннями з псевдотаблиці `EXCLUDED`.
4. **Продавці без відгуків (`NULL`):** Через `LEFT JOIN` до `review_per_order` продавці, які продали товари, але ще не отримали жодного відгуку, не зникнуть із вибірки. У них просто будуть відсутні значення агрегатних функцій (`avg_rating` = NULL, `n_reviews` = 0), і відповідно `tier` також залишиться NULL.

In [7]:
query_2_3 = text("""
ALTER TABLE olist_orders
    ADD COLUMN IF NOT EXISTS is_late BOOLEAN NOT NULL DEFAULT FALSE;

WITH delivery_flags AS (
    SELECT
        order_id,
        (
            order_delivered_customer_date IS NOT NULL
            AND order_estimated_delivery_date IS NOT NULL
            AND order_delivered_customer_date::DATE > order_estimated_delivery_date::DATE
        ) AS is_late_calc
    FROM olist_orders
)
UPDATE olist_orders AS o
SET is_late = f.is_late_calc
FROM delivery_flags AS f
WHERE f.order_id = o.order_id
RETURNING o.order_id, o.customer_id, o.is_late;
""")

with engine.begin() as conn:
    result = pd.read_sql(query_2_3, conn)
print(result.head())

                           order_id                       customer_id  is_late
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d    False
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef    False
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089    False
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82    False
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c    False


**Пояснення 2.3:**  
Запит використовує `UPDATE ... FROM`, щоб оновити колонку `is_late` на основі розрахунків із CTE `delivery_flags`. Запит є ідемпотентним: скільки б разів ми його не запускали, прапорець `is_late` завжди відображатиме актуальну різницю між фактичною та плановою датою доставки, а не просто "дописуватиме" значення наосліп.

In [8]:
query_2_4 = text("""
DELETE FROM seller_alerts
WHERE created_at < NOW() - INTERVAL '30 days'
RETURNING alert_id, seller_id, alert_type, created_at;
""")

with engine.begin() as conn:
    try:
        result = pd.read_sql(query_2_4, conn)
        print(result)
    except Exception as e:
        print("Результат порожній (немає рядків для повернення).")

Empty DataFrame
Columns: [alert_id, seller_id, alert_type, created_at]
Index: []


**Пояснення 2.4:**  
Результат запиту є порожнім, тому що ми щойно створили алерт (у завданні 2.1) із поточним часом (`NOW()`). Відповідно, у таблиці немає жодного запису, який був би старшим за 30 днів. `RETURNING` у цьому випадку корисний для моніторингу: якщо скрипт очищення спрацює через місяць, ми одразу побачимо в логах, які саме алерти були видалені.

In [9]:
query_3_1 = text("""
DROP TABLE IF EXISTS customer_segments CASCADE;
DROP TABLE IF EXISTS olist_customer_dim CASCADE;

CREATE TABLE olist_customer_dim (
    customer_unique_id TEXT PRIMARY KEY,
    latest_customer_id TEXT REFERENCES olist_customers(customer_id),
    customer_state     CHAR(2),
    customer_city      TEXT
);

INSERT INTO olist_customer_dim (
    customer_unique_id,
    latest_customer_id,
    customer_state,
    customer_city
)
SELECT DISTINCT ON (c.customer_unique_id)
    c.customer_unique_id,
    c.customer_id AS latest_customer_id,
    c.customer_state,
    c.customer_city
FROM olist_customers AS c
LEFT JOIN olist_orders AS o
    ON o.customer_id = c.customer_id
ORDER BY
    c.customer_unique_id,
    o.order_purchase_timestamp DESC NULLS LAST,
    c.customer_id;

CREATE TABLE customer_segments (
    segment_id          BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,

    customer_unique_id  TEXT NOT NULL,
    segment_name        TEXT NOT NULL
        CHECK (segment_name IN ('VIP', 'regular', 'new', 'inactive')),

    rfm_score           SMALLINT NOT NULL
        CHECK (rfm_score BETWEEN 1 AND 5),

    recency_days        INTEGER
        CHECK (recency_days IS NULL OR recency_days >= 0),

    monetary_value      NUMERIC(12, 2) NOT NULL DEFAULT 0
        CHECK (monetary_value >= 0),

    n_orders            INTEGER NOT NULL DEFAULT 0
        CHECK (n_orders >= 0),

    assigned_at         TIMESTAMPTZ NOT NULL DEFAULT NOW(),

    UNIQUE (customer_unique_id)
);

ALTER TABLE customer_segments
    ADD CONSTRAINT fk_customer_segments_customer_unique
    FOREIGN KEY (customer_unique_id)
    REFERENCES olist_customer_dim(customer_unique_id)
    ON DELETE CASCADE;
""")

with engine.begin() as conn:
    conn.execute(query_3_1)
print("Схему customer_segments та olist_customer_dim успішно створено!")

Схему customer_segments та olist_customer_dim успішно створено!


In [10]:
query_3_2 = text("""
WITH reference_date AS (
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
order_totals AS (
    SELECT
        o.order_id,
        o.customer_id,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(oi.price + oi.freight_value), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.order_id,
        o.customer_id,
        o.order_purchase_timestamp
),
customer_stats AS (
    SELECT
        c.customer_unique_id,
        COUNT(DISTINCT ot.order_id) AS n_orders,
        COALESCE(SUM(ot.order_total), 0)::NUMERIC(12, 2) AS monetary_value,
        MAX(ot.order_date) AS last_order_date
    FROM olist_customer_dim AS cd
    JOIN olist_customers AS c
        ON c.customer_unique_id = cd.customer_unique_id
    LEFT JOIN order_totals AS ot
        ON ot.customer_id = c.customer_id
    GROUP BY c.customer_unique_id
),
segmented AS (
    SELECT
        cs.customer_unique_id,
        cs.n_orders,
        cs.monetary_value,
        CASE
            WHEN cs.last_order_date IS NULL THEN NULL
            ELSE (rd.as_of_date - cs.last_order_date)::INTEGER
        END AS recency_days,
        CASE
            WHEN cs.monetary_value >= 1000 THEN 'VIP'
            WHEN cs.last_order_date IS NOT NULL
                 AND (rd.as_of_date - cs.last_order_date)::INTEGER <= 90
                 AND cs.n_orders = 1 THEN 'new'
            WHEN cs.n_orders >= 2 OR cs.monetary_value >= 100 THEN 'regular'
            ELSE 'inactive'
        END AS segment_name,
        CASE
            WHEN cs.monetary_value >= 1000 THEN 5
            WHEN cs.monetary_value >= 500  THEN 4
            WHEN cs.n_orders >= 2          THEN 3
            WHEN cs.n_orders = 1           THEN 2
            ELSE 1
        END::SMALLINT AS rfm_score
    FROM customer_stats AS cs
    CROSS JOIN reference_date AS rd
)
INSERT INTO customer_segments (
    customer_unique_id,
    segment_name,
    rfm_score,
    recency_days,
    monetary_value,
    n_orders
)
SELECT
    customer_unique_id,
    segment_name,
    rfm_score,
    recency_days,
    monetary_value,
    n_orders
FROM segmented
ON CONFLICT (customer_unique_id) DO UPDATE
SET segment_name   = EXCLUDED.segment_name,
    rfm_score      = EXCLUDED.rfm_score,
    recency_days   = EXCLUDED.recency_days,
    monetary_value = EXCLUDED.monetary_value,
    n_orders       = EXCLUDED.n_orders,
    assigned_at    = NOW()
RETURNING
    customer_unique_id,
    segment_name,
    rfm_score,
    recency_days,
    monetary_value,
    n_orders;
""")

with engine.begin() as conn:
    result = pd.read_sql(query_3_2, conn)
print(result.head())

                 customer_unique_id segment_name  rfm_score  recency_days  \
0  0000366f3b9a7992bf8c76cfdf3221e2      regular          2           161   
1  0000b849f77a49e4a4ce2b2a4ca5be3f     inactive          2           164   
2  0000f46a3911fa3c0805444483337064     inactive          2           587   
3  0000f6ccb0745a6a4b88665a16c9f078     inactive          2           371   
4  0004aac84e0df4da2b147fca70cf8255      regular          2           338   

   monetary_value  n_orders  
0          141.90         1  
1           27.19         1  
2           86.22         1  
3           43.62         1  
4          196.89         1  


In [11]:
query_3_3 = text("""
SELECT
    conname,
    contype,
    pg_get_constraintdef(oid) AS definition
FROM pg_constraint
WHERE conrelid = 'customer_segments'::regclass
ORDER BY conname;
""")

with engine.begin() as conn:
    result = pd.read_sql(query_3_3, conn)
print(result)

                                    conname contype  \
0  customer_segments_customer_unique_id_key       u   
1    customer_segments_monetary_value_check       c   
2          customer_segments_n_orders_check       c   
3                    customer_segments_pkey       p   
4      customer_segments_recency_days_check       c   
5         customer_segments_rfm_score_check       c   
6      customer_segments_segment_name_check       c   
7      fk_customer_segments_customer_unique       f   

                                          definition  
0                        UNIQUE (customer_unique_id)  
1           CHECK ((monetary_value >= (0)::numeric))  
2                            CHECK ((n_orders >= 0))  
3                           PRIMARY KEY (segment_id)  
4  CHECK (((recency_days IS NULL) OR (recency_day...  
5    CHECK (((rfm_score >= 1) AND (rfm_score <= 5)))  
6  CHECK ((segment_name = ANY (ARRAY['VIP'::text,...  
7  FOREIGN KEY (customer_unique_id) REFERENCES ol...  


**Пояснення до Завдання 3:**

1. **Чому `customer_segments` використовує `customer_unique_id`:** У датасеті Olist `customer_id` генерується наново для кожного окремого замовлення (фактично це ідентифікатор транзакції/доставки). `customer_unique_id` ідентифікує реальну фізичну людину. Розраховувати RFM (який базується на частоті покупок) за `customer_id` безглуздо, бо частота завжди дорівнюватиме 1.
2. **Чому `olist_customer_dim` потрібна для FK:** Щоб встановити зовнішній ключ (`FOREIGN KEY`), цільова таблиця повинна мати гарантовано унікальні значення у відповідній колонці (`PRIMARY KEY` або `UNIQUE`). Сира таблиця `olist_customers` містить дублікати `customer_unique_id`, тому ми створили дедуплікований вимір (dimension table), який слугує надійним довідником унікальних клієнтів.
3. **Чому `reference_date` краще брати з датасету:** Olist — це історичний датасет (дані закінчуються у 2018 році). Якби ми використали `CURRENT_DATE`, розрахунок `recency` для всіх клієнтів показав би понад 2000 днів давності, і 100% бази автоматично потрапило б до сегмента 'inactive'. Динамічна дата на основі останнього замовлення зберігає аналітичну цінність даних.
4. **Як `ON CONFLICT` робить сегментацію повторюваною:** У production-середовищі такі скрипти запускаються за розкладом (наприклад, щоночі). `ON CONFLICT` дозволяє не видаляти всю таблицю перед кожним запуском і не падати з помилкою через дублювання ключів. Натомість база просто оновлює існуючий рядок клієнта новими даними RFM-сегментації.

In [12]:
query_4_1 = text("""
SELECT
    p.product_category_name,
    s.seller_state,
    COUNT(DISTINCT oi.order_id) AS n_orders,
    SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS revenue
FROM olist_order_items AS oi
JOIN olist_orders AS o
    ON o.order_id = oi.order_id
JOIN olist_products AS p
    ON p.product_id = oi.product_id
JOIN olist_sellers AS s
    ON s.seller_id = oi.seller_id
WHERE o.order_status = 'delivered'
GROUP BY p.product_category_name, s.seller_state
ORDER BY revenue DESC NULLS LAST
LIMIT 10;
""")

with engine.begin() as conn:
    result = pd.read_sql(query_4_1, conn)
print(result)

    product_category_name seller_state  n_orders     revenue
0         cama_mesa_banho           SP      8214  1070048.14
1      relogios_presentes           SP      4494  1016168.80
2            beleza_saude           SP      5684   790967.66
3           esporte_lazer           SP      4772   680165.54
4        moveis_decoracao           SP      4597   605650.63
5              cool_stuff           SP      2621   484196.53
6   utilidades_domesticas           SP      4021   471810.66
7              automotivo           SP      3018   438833.22
8  informatica_acessorios           SP      2783   388506.42
9                   bebes           SP      1981   339129.74


**Business-question:** Які категорії товарів і штати продавців дають найбільший revenue у доставлених (delivered) замовленнях?

**Interpretation:** Запит використовує `INNER JOIN` на 4 таблицях (замовлення, позиції, товари, продавці), відкидаючи будь-які записи, що не мають відповідників у всіх цих сутностях. Результат показує найприбутковіші зв'язки "категорія-штат", що допомагає зрозуміти, у яких регіонах зосереджені найуспішніші продавці конкретних ніш (наприклад, лідерами зазвичай є продавці зі штату SP).

In [13]:
query_4_2 = text("""
WITH order_totals AS (
    SELECT
        o.order_id,
        c.customer_unique_id,
        SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.order_id,
        c.customer_unique_id
),
review_per_order AS (
    SELECT
        rv.order_id,
        AVG(rv.review_score)::NUMERIC(3, 2) AS order_review_score
    FROM olist_order_reviews AS rv
    WHERE rv.review_score IS NOT NULL
    GROUP BY rv.order_id
),
customer_stats AS (
    SELECT
        ot.customer_unique_id,
        COUNT(DISTINCT ot.order_id) AS n_orders,
        COALESCE(SUM(ot.order_total), 0)::NUMERIC(12, 2) AS total_spend,
        AVG(rpo.order_review_score)::NUMERIC(3, 2) AS avg_review_score
    FROM order_totals AS ot
    LEFT JOIN review_per_order AS rpo
        ON rpo.order_id = ot.order_id
    GROUP BY ot.customer_unique_id
)
SELECT
    cd.customer_unique_id,
    cd.customer_state,
    COALESCE(cs.n_orders, 0) AS n_orders,
    COALESCE(cs.total_spend, 0)::NUMERIC(12, 2) AS total_spend,
    COALESCE(cs.avg_review_score, 0)::NUMERIC(3, 2) AS avg_review_score
FROM olist_customer_dim AS cd
LEFT JOIN customer_stats AS cs
    ON cs.customer_unique_id = cd.customer_unique_id
ORDER BY total_spend DESC NULLS LAST
LIMIT 20;
""")

with engine.begin() as conn:
    result = pd.read_sql(query_4_2, conn)
print(result)

                  customer_unique_id customer_state  n_orders  total_spend  \
0   0a0a92112bd4c708ca5fde585afaa872             RJ         1     13664.08   
1   da122df9eeddfedc1dc1f5349a1a690c             RJ         2      7571.63   
2   763c8b1c9c68a0229c42c9fc6f662b93             ES         1      7274.88   
3   dc4802a71eae9be1dd28f5d788ceb526             MS         1      6929.31   
4   459bef486812aa25204be022145caa62             ES         1      6922.21   
5   ff4159b92c40ebe40454e3e6a7c35ed6             SP         1      6726.66   
6   4007669dec559734d6f53e029e360987             MG         1      6081.54   
7   5d0a2980b292d049061542014e8960bf             GO         1      4809.44   
8   eebb5dda148d3893cdaf5b5ca3040ccb             SP         1      4764.34   
9   48e1ac109decbb87765a3eade6854098             PB         1      4681.78   
10  c8460e4251689ba205045f3ea17884a1             RS         4      4655.88   
11  edde2314c6c30e864a128ac95d6b2112             SP         1   

**Business-question:** Які customer-level features (фічі на рівні реального покупця) можна побудувати для аналітики та ML?

**Interpretation:** Цей запит збирає профіль унікального клієнта, розраховуючи його загальні витрати та середню оцінку, яку він залишав.
*Критично важливий момент:* СТЕ `review_per_order` попередньо агрегує відгуки на рівні замовлення. Це захищає метрику від штучного множення (якби ми напряму приєднали відгуки до `order_items`, одне замовлення з 5 товарами і 1 відгуком згенерувало б 5 однакових оцінок для розрахунку середнього).
Функція `COALESCE` гарантує, що клієнти без замовлень (або витрат) отримають `0`, а не `NULL`, що ідеально підходить для ML-моделей.

In [14]:
query_4_3 = text("""
WITH customer_states AS (
    SELECT customer_state AS state, COUNT(*) AS n_customers
    FROM olist_customers
    GROUP BY customer_state
), seller_states AS (
    SELECT seller_state AS state, COUNT(*) AS n_sellers
    FROM olist_sellers
    GROUP BY seller_state
)
SELECT
    COALESCE(c.state, s.state) AS state,
    c.n_customers,
    s.n_sellers,
    CASE
        WHEN c.state IS NOT NULL AND s.state IS NOT NULL THEN 'both'
        WHEN c.state IS NOT NULL THEN 'customers_only'
        ELSE 'sellers_only'
    END AS state_presence
FROM customer_states AS c
FULL OUTER JOIN seller_states AS s
    ON s.state = c.state
ORDER BY state;
""")

with engine.begin() as conn:
    result = pd.read_sql(query_4_3, conn)
print(result)

   state  n_customers  n_sellers  state_presence
0     AC           81        1.0            both
1     AL          413        NaN  customers_only
2     AM          148        1.0            both
3     AP           68        NaN  customers_only
4     BA         3380       19.0            both
5     CE         1336       13.0            both
6     DF         2140       30.0            both
7     ES         2033       23.0            both
8     GO         2020       40.0            both
9     MA          747        1.0            both
10    MG        11635      244.0            both
11    MS          715        5.0            both
12    MT          907        4.0            both
13    PA          975        1.0            both
14    PB          536        6.0            both
15    PE         1652        9.0            both
16    PI          495        1.0            both
17    PR         5045      349.0            both
18    RJ        12852      171.0            both
19    RN          48

**Business-question:** Які штати представлені тільки серед клієнтів, тільки серед продавців або в обох групах?

**Interpretation:** Завдяки `FULL OUTER JOIN` ми бачимо повну картину покриття платформи. Запит не відкидає штати, де є лише покупці (але немає жодного зареєстрованого продавця Olist), і навпаки. Це класичний інструмент для виявлення розбіжностей між двома множинами (наприклад, для планування регіональної експансії залучення продавців).

In [15]:
query_4_4 = text("""
SELECT
    s1.seller_id AS seller_a,
    s2.seller_id AS seller_b,
    s1.seller_state,
    s1.seller_city AS city_a,
    s2.seller_city AS city_b
FROM olist_sellers AS s1
JOIN olist_sellers AS s2
    ON s1.seller_state = s2.seller_state
   AND s1.seller_id < s2.seller_id
   AND s1.seller_city IS DISTINCT FROM s2.seller_city
ORDER BY s1.seller_state, seller_a, seller_b
LIMIT 20;
""")

with engine.begin() as conn:
    result = pd.read_sql(query_4_4, conn)
print(result)

                            seller_a                          seller_b  \
0   1444c08e64d55fb3c25f0f09c07ffcf2  2b402d5dc42554061f8ea98d1916f148   
1   1444c08e64d55fb3c25f0f09c07ffcf2  398cb257329ef7af7f1943a8974a3cbc   
2   1444c08e64d55fb3c25f0f09c07ffcf2  4221a7df464f1fe2955934e30ff3a5a1   
3   1444c08e64d55fb3c25f0f09c07ffcf2  43753b27d77860f1654aa72e251a7878   
4   1444c08e64d55fb3c25f0f09c07ffcf2  4aba391bc3b88717ce08eb11e44937b2   
5   1444c08e64d55fb3c25f0f09c07ffcf2  4fb41dff7c50136976d1a5cf004a42e2   
6   1444c08e64d55fb3c25f0f09c07ffcf2  651530bf5c607240ccdd89a30c9c9712   
7   1444c08e64d55fb3c25f0f09c07ffcf2  659e8466eb3ff1b0e8740d74fb7bbedd   
8   1444c08e64d55fb3c25f0f09c07ffcf2  75d34ebb1bd0bd7dde40dd507b8169c3   
9   1444c08e64d55fb3c25f0f09c07ffcf2  a3dd39f583bc80bd8c5901c95878921e   
10  1444c08e64d55fb3c25f0f09c07ffcf2  b00af24704019bd2e1b335e70ad11f7c   
11  1444c08e64d55fb3c25f0f09c07ffcf2  c72de06d72748d1a0dfb2125be43ba63   
12  1444c08e64d55fb3c25f0f09c07ffcf2  

**Business-question:** Які seller-пари можуть бути кандидатами для регіонального порівняння конкуренції в межах одного штату (але в різних містах)?

**Interpretation:** Таблиця `olist_sellers` з'єднується сама із собою (`SELF JOIN`). Умова `s1.seller_id < s2.seller_id` гарантує, що ми не порівнюємо продавця із самим собою і що пара A-B не дублюється як B-A. Умова `IS DISTINCT FROM` забезпечує, що вони знаходяться в різних містах.

In [16]:
query_4_5 = text("""
SELECT
    o.order_id,
    o.customer_id,
    o.order_status,
    o.order_purchase_timestamp
FROM olist_orders AS o
LEFT JOIN olist_order_reviews AS rv
    ON rv.order_id = o.order_id
WHERE rv.review_row_id IS NULL
ORDER BY o.order_purchase_timestamp
LIMIT 20;
""")

with engine.begin() as conn:
    result = pd.read_sql(query_4_5, conn)
print(result)

                            order_id                       customer_id  \
0   cda873529ca7ab71f677d5ec11a40304  76c74aaff2f3f7355f46d9818ad092b8   
1   5da2004d85f2828349d887c989da2adc  95f592086150f0c95d3082aea047b816   
2   02190241f7190a1f3c7e0df95a749c6a  55c9044401ffab0c717f24405e2e042f   
3   0efd0bc268d34da3f01f4ff25a6d4335  95446917717bb58d553d107d0f1668f6   
4   c776863a93dc0740c6e7d78104b21413  1710b798ebdc5db6665c815b90a64999   
5   d730d8d0ea3149120703933fdf98b2d4  ce2a66464da8b80dedd7639a6e489a58   
6   05cd156f0f47579a36d7a52724fc31b8  976224a95faf328b17654b1b1411d5e1   
7   719b940ab2f1e351ffb14a0a02ffe3c4  ede9a1daf58d256c0d648ef7e1479010   
8   fbd82b1fe98967de0db29b6cf134d00a  45f5c9d984dbfa2105e3544cc85133bf   
9   171398a0cd843c65ac78aee9e05a1f49  f0244902c07b7c0799e468218579a925   
10  fcda3cc8210e746326864eb933641ccf  fee7cf49127308e2b5402c9984b6f1f6   
11  58a4317c14bcba43d0cf161f17e9db2b  e7f5b701d99a5eb91e57a970a75640f0   
12  ee55010a370e987f203499f435a59847  

**Business-question:** Які замовлення не мають жодного залишеного відгуку?

**Interpretation:** Це класичний патерн Anti-join. Ми беремо всі замовлення через `LEFT JOIN`, а потім фільтруємо (`WHERE rv.review_row_id IS NULL`), залишаючи лише ті рядки, яким база не змогла знайти відповідника в таблиці відгуків. Цей підхід працює швидше і безпечніше (стійкий до NULL), ніж використання оператора `NOT IN`.

In [17]:
query_5_1 = text("""
SELECT
    p.product_category_name,
    COUNT(DISTINCT oi.order_id) AS n_orders,
    SUM(oi.price)::NUMERIC(12, 2) AS gross_revenue,
    SUM(oi.freight_value)::NUMERIC(12, 2) AS freight,
    AVG(oi.price)::NUMERIC(8, 2) AS avg_item_price
FROM olist_order_items AS oi
JOIN olist_products AS p
    ON p.product_id = oi.product_id
GROUP BY p.product_category_name
HAVING SUM(oi.price) > 10000
ORDER BY gross_revenue DESC NULLS LAST;
""")

with engine.begin() as conn:
    result = pd.read_sql(query_5_1, conn)
print(result.head(10))

    product_category_name  n_orders  gross_revenue    freight  avg_item_price
0            beleza_saude      8836     1258681.34  182566.73          130.16
1      relogios_presentes      5624     1205005.68  100535.93          201.14
2         cama_mesa_banho      9417     1036988.68  204693.04           93.30
3           esporte_lazer      7720      988048.97  168607.51          114.34
4  informatica_acessorios      6689      911954.32  147318.08          116.51
5        moveis_decoracao      6449      729762.49  172749.30           87.56
6              cool_stuff      3632      635290.85   84039.10          167.36
7   utilidades_domesticas      5884      632248.66  146149.11           90.79
8              automotivo      3897      592720.11   92664.21          139.96
9      ferramentas_jardim      3518      485256.46   98962.75          111.63


In [18]:
query_5_2 = text("""
WITH order_totals AS (
    SELECT
        o.order_id,
        c.customer_state,
        SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
    JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.order_id,
        c.customer_state
)
SELECT
    customer_state,
    COUNT(*) AS n_orders,
    SUM(order_total)::NUMERIC(12, 2) AS gmv,
    (SUM(order_total) / NULLIF(COUNT(*), 0))::NUMERIC(8, 2) AS aov
FROM order_totals
GROUP BY customer_state
HAVING COUNT(*) >= 100
ORDER BY aov DESC NULLS LAST;
""")

with engine.begin() as conn:
    result = pd.read_sql(query_5_2, conn)
print(result.head(10))

  customer_state  n_orders        gmv     aov
0             PB       532  140987.81  265.01
1             AL       411   96229.40  234.13
2             RO       247   57558.02  233.03
3             PA       970  217647.11  224.38
4             TO       279   61354.42  219.91
5             PI       493  108132.28  219.34
6             SE       345   73032.32  211.69
7             RN       482  101895.08  211.40
8             CE      1327  275606.30  207.69
9             MT       903  186168.96  206.17


In [19]:
query_5_3 = text("""
SELECT
    c.customer_unique_id,
    COUNT(DISTINCT o.order_id) AS n_orders,
    SUM(oi.price)::NUMERIC(12, 2) AS total_spend,
    STRING_AGG(
        DISTINCT COALESCE(p.product_category_name, '(unknown)'),
        ', ' ORDER BY COALESCE(p.product_category_name, '(unknown)')
    ) AS categories
FROM olist_customers AS c
JOIN olist_orders AS o
    ON o.customer_id = c.customer_id
JOIN olist_order_items AS oi
    ON oi.order_id = o.order_id
LEFT JOIN olist_products AS p
    ON p.product_id = oi.product_id
GROUP BY c.customer_unique_id
HAVING COUNT(DISTINCT o.order_id) >= 1
ORDER BY total_spend DESC NULLS LAST
LIMIT 20;
""")

with engine.begin() as conn:
    result = pd.read_sql(query_5_3, conn)
print(result)

                  customer_unique_id  n_orders  total_spend  \
0   0a0a92112bd4c708ca5fde585afaa872         1     13440.00   
1   da122df9eeddfedc1dc1f5349a1a690c         2      7388.00   
2   763c8b1c9c68a0229c42c9fc6f662b93         1      7160.00   
3   dc4802a71eae9be1dd28f5d788ceb526         1      6735.00   
4   459bef486812aa25204be022145caa62         1      6729.00   
5   ff4159b92c40ebe40454e3e6a7c35ed6         1      6499.00   
6   4007669dec559734d6f53e029e360987         1      5934.60   
7   eebb5dda148d3893cdaf5b5ca3040ccb         1      4690.00   
8   5d0a2980b292d049061542014e8960bf         1      4599.90   
9   48e1ac109decbb87765a3eade6854098         1      4590.00   
10  a229eba70ec1c2abef51f04987deb7a5         1      4400.00   
11  edde2314c6c30e864a128ac95d6b2112         1      4399.87   
12  fa562ef24d41361e476e748681810e1e         1      4099.99   
13  c8460e4251689ba205045f3ea17884a1         4      4080.00   
14  ca27f3dac28fb1063faddd424c9d95fa         1      405

**Різниця між WHERE та HAVING (на прикладі Запиту 5.1):**

*   **`WHERE`** працює на рівні окремих записів **до того, як вони будуть згруповані**[cite: 8]. Наприклад, якби ми додали `WHERE oi.price > 50`, база спочатку відкинула б усі дешеві товари, а вже потім почала рахувати загальні суми для залишених категорій.
*   **`HAVING`** працює на рівні вже сформованих груп **після виконання агрегації**[cite: 8]. У запиті 5.1 ми використовуємо `HAVING SUM(oi.price) > 10000` — ця умова перевіряє вже фінальну, пораховану загальну суму для всієї категорії, і відкидає ті категорії (тобто групи), які не досягли цього порогу прибутку. Використати агрегатну функцію на кшталт `SUM()` у блоці `WHERE` синтаксично неможливо.

In [20]:
query_6_1 = text("""
SELECT customer_state AS state
FROM olist_customers
INTERSECT
SELECT seller_state AS state
FROM olist_sellers
ORDER BY state;
""")

with engine.begin() as conn:
    result = pd.read_sql(query_6_1, conn)
print(result)

   state
0     AC
1     AM
2     BA
3     CE
4     DF
5     ES
6     GO
7     MA
8     MG
9     MS
10    MT
11    PA
12    PB
13    PE
14    PI
15    PR
16    RJ
17    RN
18    RO
19    RS
20    SC
21    SE
22    SP


**Чому обрано `INTERSECT`:**
Цей оператор ідеально підходить для математичного перетину множин. Він автоматично прибирає дублікати й залишає виключно ті штати, які одночасно присутні в обох таблицях. Використання `UNION` тут змішало б усе в одну купу, а `EXCEPT` показав би лише різницю.

In [21]:
query_6_2 = text("""
WITH ranked_orders AS (
    SELECT
        c.customer_unique_id,
        o.customer_id,
        o.order_id,
        o.order_purchase_timestamp,
        ROW_NUMBER() OVER (
            PARTITION BY c.customer_unique_id
            ORDER BY o.order_purchase_timestamp DESC, o.order_id DESC
        ) AS rn
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
)
SELECT
    customer_unique_id,
    customer_id,
    order_id,
    order_purchase_timestamp
FROM ranked_orders
WHERE rn = 1
ORDER BY customer_unique_id
LIMIT 20;
""")

with engine.begin() as conn:
    result = pd.read_sql(query_6_2, conn)
print(result)

                  customer_unique_id                       customer_id  \
0   0000366f3b9a7992bf8c76cfdf3221e2  fadbb3709178fc513abc1b2670aa1ad2   
1   0000b849f77a49e4a4ce2b2a4ca5be3f  4cb282e167ae9234755102258dd52ee8   
2   0000f46a3911fa3c0805444483337064  9b3932a6253894a02c1df9d19004239f   
3   0000f6ccb0745a6a4b88665a16c9f078  914991f0c02ef0843c0e7010c819d642   
4   0004aac84e0df4da2b147fca70cf8255  47227568b10f5f58a524a75507e6992c   
5   0004bd2a26a76fe21f786e4fbd80607f  4a913a170c26e3c8052ed0202849b5a8   
6   00050ab1314c0e55a6ca13cf7181fecf  d2509c13692836fc0449e88cf9eb4858   
7   00053a61a98854899e70ed204dd4bafe  a81ebb9b32f102298c0c89635b4b3154   
8   0005e1862207bf6ccc02e4228effd9a0  3b37fb626fdf46cd99d37ec62afa88ff   
9   0005ef4cd20d2893f0d9fbd94d3c0d97  c59e8ff99836e90d8b457d4122dc34e9   
10  0006fdc98a402fceb4eb0ee528f6a8d4  2f29573c8cac5a7be11c5b649078f944   
11  00082cbe03e478190aadbea78542e933  9dec2d47e95677b616efe366f8bf9fb8   
12  00090324bbad0e9342388303bb71ba0a  

### Reflection

**1. Найскладніший JOIN:**
Найбільше уваги вимагав `FULL OUTER JOIN`. Його специфіка полягає в тому, що після з'єднання потрібно постійно використовувати `COALESCE` для ключів (наприклад, штатів), інакше штати, які є лише в правій таблиці, виводилися б як `NULL`.

**2. Найскладніший business-question:**
Важко було придумати реалістичне бізнес-запитання для `SELF JOIN`. Порівняння продавців з одного штату, але різних міст, виглядає трохи штучно, хоча й має сенс для аналізу регіональної конкуренції.

**3. Найкорисніша фіча для production-ETL:**
Безперечно, `ON CONFLICT DO UPDATE`. Ідемпотентність — це база для будь-якого ETL. Можливість перезапускати скрипти скільки завгодно разів без страху отримати помилку дублювання ключів або розмножити дані робить пайплайни стабільними.

**4. Використання window functions у ML-проєктах:**
У реальному ML я б використовував їх для створення часових фічей: `LAG()` для розрахунку кількості днів між поточною та попередньою покупкою, або `SUM() OVER (ORDER BY date)` для розрахунку кумулятивних (накопичувальних) витрат клієнта на момент кожної транзакції.

**5. Data-quality проблеми:**
Окрім кумедної одруківки `lenght` у назвах колонок, кидається в очі наявність замовлень взагалі без відгуків (що нормально для бізнесу, але вимагає уваги в SQL через `NULL`), а також те, що один і той самий відгук технічно дублюється для всіх товарів у замовленні.

**6. `customer_unique_id` замість `customer_id`:**
У логіці Olist `customer_id` — це одноразовий ідентифікатор сесії/доставки. Якби ми будували ML-фічі за ним, кожен клієнт здавався б новим, а частота покупок (Frequency) завжди дорівнювала б 1. `customer_unique_id` дозволяє "склеїти" всі покупки реальної людини.

**7. Грануляність `seller_id × order_id`:**
Відгуки (`review_score`) в Olist ставляться всьому замовленню в цілому. Якщо покупець замовив 5 ручок у одного продавця, це буде 5 записів у `order_items`. Прямий JOIN розмножив би один відгук 5 разів, що штучно завищило б кількість відгуків (`n_reviews`) і спотворило б середній рейтинг продавця.